# 03 — Donor-marker calibration

This notebook inspects the production `controls` and `calibrate` artifacts. Calibration uses registered mutually exclusive references to estimate donor-local marker background and uncertainty. It does not fit a RESTORE/GMM maximum, normalize by positive-cell frequency, or replace the original intensity scale.

In [ ]:
import pandas as pd
from IPython.display import display

from phenocycler.artifacts import StageManifest
from phenocycler.config import load_config
from phenocycler.expression import read_single_partition
from phenocycler.pipeline import RunContext, resolve_run_context, run_stage, status

CONFIG_PATH = None
cfg = load_config(CONFIG_PATH)
context: RunContext = resolve_run_context(cfg)
print(f"run_id={context.run_id}  donors={len(context.donors)}  root={context.run_root}")

In [ ]:
status_code = status(context)
print(f"status return code: {status_code}")

## Evidence semantics

Each calibrated marker keeps several deliberately distinct quantities:

- **Intensity** (`<marker>__corrected_intensity`) is the authoritative selected-expression value and retains its physical scale.
- **Log2 threshold ratio** is the signed log2 separation from the donor-marker threshold after the production `log1p` transform; zero is the threshold, positive is above it, and negative is below it.
- **Empirical tail probability** is the upper-tail null probability under clean reference controls; smaller values are stronger evidence against background.
- **Expression probability** is `1 - empirical_tail_probability`; larger values are stronger expression evidence. It is an evidence score, not a cohort frequency or an absolute cross-donor abundance.
- **State** is `positive`, `negative`, `indeterminate`, or `unavailable`. A cell is positive or negative only when its value is stable relative to the bootstrapped threshold interval and its empirical evidence agrees; otherwise uncertainty is preserved.

The model audit records control counts, contamination, threshold intervals, status, and failure reasons. Invalid donor-marker models leave measurable cells `indeterminate`; an unmeasured cell or panel-absent marker is `unavailable`. Neither becomes a guessed negative.

## Optional production execution

`controls` selects mutually exclusive background references from estimation-eligible cells. `calibrate` fits the donor-marker models there, then applies them to every measurable cell while preserving analysis eligibility for typing. Existing valid stages are checked rather than recomputed.

In [ ]:
RUN_STAGES = False

if RUN_STAGES:
    for stage_name in ("controls", "calibrate"):
        run_stage(context, stage_name)
else:
    print("Inspection only. Set RUN_STAGES=True to run production control selection and calibration.")

In [ ]:
manifest_rows = []
for stage_name in ("controls", "calibrate"):
    path = context.stage_manifest_path(stage_name)
    if path.exists():
        manifest = StageManifest.read_json(path)
        manifest_rows.append({
            "stage": stage_name,
            "method_version": manifest.method_version,
            "donors": len(manifest.completed_donors),
            "rows": manifest.output.total_rows,
            "columns": len(manifest.output.schema),
            "schema": manifest.output.schema_sha256[:12],
            "objects": manifest.output.object_id_sha256[:12],
            "content": manifest.content_id[:12],
        })
display(pd.DataFrame(manifest_rows))

## Model audits

Inspect validity and uncertainty before interpreting per-cell evidence. The calibration audit is one row per donor-marker model; the control audit records how reference controls were selected.

In [ ]:
control_audit_path = context.config.audit_dir / "reference_control_models.parquet"
calibration_audit_path = context.config.audit_dir / "marker_calibration_models.parquet"

if control_audit_path.exists():
    control_audit = pd.read_parquet(control_audit_path)
    print(f"control models: {len(control_audit):,}")
    display(control_audit.head(20))
else:
    print("Reference-control model audit is not present.")

if calibration_audit_path.exists():
    calibration_audit = pd.read_parquet(calibration_audit_path)
    print(f"calibration models: {len(calibration_audit):,}")
    display(calibration_audit.groupby("status", dropna=False).size().rename("donor_markers").to_frame())
    audit_columns = [
        column for column in (
            "donor_id", "marker", "reference", "status", "status_reason",
            "marker_alpha", "n_controls", "control_contamination_fraction",
            "threshold_raw", "threshold_ci_low_raw", "threshold_ci_high_raw"
        ) if column in calibration_audit
    ]
    display(calibration_audit.loc[:, audit_columns].head(30))
else:
    calibration_audit = pd.DataFrame()
    print("Marker-calibration model audit is not present.")

## Inspect one donor-marker pair

The evidence table is wide by design: every registered marker contributes its intensity, calibrated state, threshold-relative score, empirical probabilities, and repeated model provenance. Choose a donor and marker below to inspect the same quantities that downstream typing consumes.

In [ ]:
DONOR = context.donors[0]
evidence_manifest = context.stage_manifest_path("calibrate")

if evidence_manifest.exists():
    evidence = read_single_partition(context.config.marker_evidence_dir, DONOR)
    calibrated_markers = [
        marker.name
        for marker in context.registry.active_markers
        if f"{marker.name}__state" in evidence
    ]
    if calibrated_markers:
        MARKER = calibrated_markers[0]
        metric_suffixes = (
            "corrected_intensity", "state", "log2_threshold_ratio",
            "empirical_tail_probability", "expression_probability",
            "model_valid", "calibration_status", "threshold",
            "threshold_ci_low", "threshold_ci_high", "n_controls",
            "control_contamination_fraction"
        )
        metric_columns = [
            f"{MARKER}__{suffix}"
            for suffix in metric_suffixes
            if f"{MARKER}__{suffix}" in evidence
        ]
        display(evidence.loc[:, ["object_id", *metric_columns]].head(20))
        display(evidence[f"{MARKER}__state"].value_counts(dropna=False).rename("cells").to_frame())
        if not calibration_audit.empty:
            pair_model = calibration_audit.loc[
                calibration_audit["donor_id"].astype(str).eq(DONOR)
                & calibration_audit["marker"].astype(str).eq(MARKER)
            ]
            display(pair_model)
    else:
        print(f"No calibrated marker columns are available for donor {DONOR}.")
else:
    print("Calibration artifacts are not complete yet.")

## Handoff

Proceed to typing only after reviewing invalid and indeterminate donor-marker models—not just positive-cell counts. Hierarchical typing consumes calibrated evidence with its uncertainty intact, so missing or unreliable markers remain explicit rather than silently becoming negatives.